In [ ]:
from pathlib import Path
import sys
import subprocess
import time
import importlib.metadata
import json
import pandas as pd
from IPython.display import display

In [ ]:
ROOT = Path.cwd()
TASKS = ['a_import','b_prepare','c_fit','d_simulate','e_diagnose','f_plot','g_report']
# recreate ignored task directories and input links on a fresh checkout
for task in TASKS:
    (ROOT/task/'output').mkdir(parents=True,exist_ok=True)
(ROOT/'a_import/input').mkdir(parents=True,exist_ok=True)
links = {f'{task}/input':f'../{previous}/output' for previous,task in zip(TASKS,TASKS[1:])}
links.update({'c_fit/output/data':'../input','d_simulate/output/fit':'../input','e_diagnose/output/simulation':'../input'})
for relative,target in links.items():
    path = ROOT/relative
    if not path.is_symlink():path.symlink_to(target,target_is_directory=True)
plan = [path for task in TASKS for path in sorted((ROOT/task/'src').rglob('*.ipynb'))]
pd.DataFrame({'order':range(1,len(plan)+1),'notebook':[str(p.relative_to(ROOT)) for p in plan]})

In [ ]:
# isolated processes, with notebook source left unchanged
WORKER = r"""import json, os, sys
from pathlib import Path
import matplotlib
from IPython.core.interactiveshell import InteractiveShell
from IPython.utils.capture import capture_output

path = Path(sys.argv[1]).resolve()
os.chdir(path.parent)
matplotlib.use('Agg')
notebook = json.loads(path.read_text())
shell = InteractiveShell.instance()
for index, cell in enumerate(notebook['cells'], 1):
    with capture_output() as captured:
        result = shell.run_cell(''.join(cell['source']), store_history=False)
    error = result.error_before_exec or result.error_in_exec
    if error:
        print(f'Cell {index}:\n{captured.stdout}{captured.stderr}', file=sys.stderr)
        raise error
"""
runs = []
for path in plan:
    relative = str(path.relative_to(ROOT))
    start = time.perf_counter()
    result = subprocess.run([sys.executable, '-c', WORKER, str(path)], capture_output=True, text=True)
    status = 'passed' if result.returncode == 0 else 'failed'
    runs.append({'notebook':relative, 'seconds':time.perf_counter()-start, 'status':status})
    pd.DataFrame(runs).to_csv(ROOT/'pipeline_run.csv',index=False)
    if result.returncode:
        raise RuntimeError(f'{relative}\n{result.stdout}\n{result.stderr}')
display(pd.DataFrame(runs))

In [ ]:
packages = ['numpy','pandas','scipy','scikit-learn','matplotlib','seaborn','ipython']
versions = pd.DataFrame([{'package':p,'version':importlib.metadata.version(p)} for p in packages])
versions.to_csv(ROOT/'environment_versions.csv',index=False)
display(versions)